In [3]:
import redis

# Configurações de conexão
# REDIS_HOST ='10.101.120.4'
REDIS_HOST = "localhost"
REDIS_PORT = 6379
# Cria uma conexão com o Redis
r = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
)

result = r.ping()
print(result)

True


In [4]:
import redis
import os
from typing import Optional
from dotenv import load_dotenv
import logging

# Carrega as variáveis de ambiente do arquivo .env
load_dotenv()

if not os.getenv('AZURE_REDIS_HOST'):
    print("❌ Erro: Variável de ambiente REDIS_HOST não encontrada")


class RedisManager:
    """
    Gerenciador unificado para Redis - Local e Cloud
    """
    
    def __init__(self):
        self.redis_client = None
        self.environment = self._detect_environment()
        
    def _detect_environment(self) -> str:

        azure_indicators = [
            'AZURE_REDIS_HOST',
            'AZURE_REDIS_PASSWORD',
            'AZURE_REDIS_CONNECTION_STRING',
            'REDIS_AZURE_URL',
        ]
        
        # Se qualquer uma das variáveis de ambiente do Azure estiver presente,
        # assumimos que estamos em um ambiente Azure.
        if any(os.getenv(indicator) for indicator in azure_indicators):
            return 'azure'
        
        # Se não houver indicadores do Azure, assume que o ambiente é local.
        return 'local'
    
    def connect(self) -> Optional[redis.Redis]:
        """
        Conecta ao Redis baseado no ambiente detectado
        """
        logging.info(f"🔍 Ambiente detectado: {self.environment}")

        try:
            if self.environment == 'local':
                self.redis_client = self._connect_local()
            elif self.environment == 'azure':
                self.redis_client = self._connect_azure()
            else:
                logging.error("❌ Ambiente não reconhecido, tentando local como fallback...")
                self.redis_client = self._connect_local()

            if self.redis_client and self.redis_client.ping():
                logging.info(f"✅ Conexão bem-sucedida ao Redis no ambiente '{self.environment}'")
                return self.redis_client
            else:
                logging.error("❌ Falha ao conectar ao Redis ou cliente não disponível.")
                return None
                
        except (redis.exceptions.ConnectionError, redis.exceptions.AuthenticationError) as e:
            logging.error(f"❌ Erro de conexão ou autenticação com o Redis: {e}")
            return None
        except Exception as e:
            logging.error(f"❌ Erro inesperado ao conectar ao Redis: {e}")
            return None

    def _connect_local(self) -> redis.Redis:
        """
        Cria e retorna uma instância do cliente Redis para um ambiente local.
        
        Usa variáveis de ambiente para a configuração, com valores padrão
        para facilitar o uso em desenvolvimento.
        """
        host = os.getenv('REDIS_HOST', 'localhost')
        port = int(os.getenv('REDIS_PORT', 10000))
        # db = int(os.getenv('REDIS_DB', 0))

        logging.info(f"🔄 Conectando ao Redis LOCAL em {host}:{port}...")
        
        return redis.Redis(
            host=host,
            port=port,
            # db=db,
            # decode_responses=True,
            # socket_connect_timeout=2,
            # socket_timeout=2,
        )
    
    def _connect_azure(self) -> Optional[redis.Redis]:
        """Conecta ao Azure Redis Cache"""
        
        # Prioridade 1: Conexão por URL
        # azure_url = os.getenv('REDIS_AZURE_URL')
        # if azure_url:
        #     logging.info("🔄 Conectando via Azure Redis URL...")
        #     return redis.from_url(
        #         azure_url,
        #         decode_responses=True,
        #         socket_connect_timeout=10,
        #         ssl_cert_reqs=None
        #     )
        
        # # Prioridade 2: Conexão por Connection String
        # connection_string = os.getenv('AZURE_REDIS_CONNECTION_STRING')
        # if connection_string:
        #     logging.info("🔄 Conectando via Azure Redis Connection String...")
        #     return redis.from_url(
        #         f"rediss://{connection_string}",
        #         decode_responses=True,
        #         socket_connect_timeout=10,
        #         ssl_cert_reqs=None
        #     )
            
        # Prioridade 3: Conexão manual
        host = os.getenv('AZURE_REDIS_HOST') or os.getenv('REDIS_HOST')
        # password = os.getenv('AZURE_REDIS_PASSWORD') or os.getenv('REDIS_PASSWORD')
        port = int(os.getenv('AZURE_REDIS_PORT', os.getenv('REDIS_PORT', 10000)))
        
        if host and port:
            logging.info(f"🔄 Conectando ao Azure Redis em {host}:{port}...")
            return redis.Redis(
                host=host,
                port=port,
                # password=password,
                # ssl=True,
                # ssl_cert_reqs=None,
                # decode_responses=True,
                # socket_connect_timeout=10,
                # health_check_interval=30
            )
        
        logging.error("❌ Nenhuma configuração válida do Azure Redis foi encontrada.")
        return None



if __name__ == "__main__":

    # manager = RedisManager()  
    # redis_client = manager.connect()  # Local ou Azure
    # print(redis_client)

    manager = RedisManager()  
    redis_client = manager.connect()

    if redis_client:
        print("Conexão com o Redis estabelecida com sucesso!")
        # Exemplo de uso
        # redis_client.set("chave_teste", "valor_teste")
        # print(redis_client.get("chave_teste"))
    else:
        print("Falha ao conectar com o Redis.")

❌ Erro: Variável de ambiente REDIS_HOST não encontrada
Conexão com o Redis estabelecida com sucesso!


In [5]:
import redis
import json
import os
import hashlib
from datetime import datetime
from typing import Optional, Dict, List, Any
import re


class SQLCache:
    """
    Cache simples para perguntas e queries SQL
    """

    def __init__(self, redis_client: redis.Redis):
        """
        Inicializa o cache com um cliente Redis já conectado.
        """
        self.redis = redis_client
        if not self.redis or not self.redis.ping():
            raise Exception("Erro: Cliente Redis não conectado ou inválido!")
            
    def normalize_question(self, question: str) -> str:
        """Normaliza pergunta para busca"""
        normalized = re.sub(r'[^\w\s]', '', question.lower())
        return ' '.join(normalized.split())

    def hash_question(self, question: str) -> str:
        """Cria hash da pergunta normalizada"""
        normalized = self.normalize_question(question)
        return hashlib.md5(normalized.encode()).hexdigest()[:12]  # 12 chars

    def store_qa(self, question: str, querye: str, metadata: dict = None) -> str:
        """
        Armazena pergunta, query e informações adicionais no cache.
        Apenas 'question' e 'querye' são obrigatórias.
        
        Args:
            question (str): A pergunta do utilizador.
            querye (str): A query SQL gerada.
            **metadata (Any): Campos adicionais como fonte, departamento, etc.
        
        Returns:
            str: O hash único da pergunta.
        """

        if metadata is None:
            metadata = {}

        if querye is None:
            querye = ""
        # Cria um hash único para a pergunta
        question_hash = self.hash_question(question)
        
        # Estrutura principal com as informações obrigatórias e geradas
        default_metadata = {
            "id": None,  # ID padrão baseado no hash
            "fonte": None,
            "departamento": None,
            "explanation_query": None,
            "python_viz": None,
            "data_criacao": datetime.now().strftime("%d/%m/%Y/"),
            "data_atualizacao": datetime.now().strftime("%d/%m/%Y/"),
            "status": None,
            "usuario_criacao": None,
        }

        processed_metadata = {**default_metadata, **metadata}

        entry = {
            "question": question,
            "querye": querye,
            **processed_metadata,  # Adiciona todos os metadados processados
        }
        
        # Armazena a entrada como uma string JSON no Redis.
        # Usa SETEX para definir um tempo de expiração (30 dias)
        key = f"sql_cache:{question_hash}"
        self.redis.setex(key, 86400 * 30, json.dumps(entry, ensure_ascii=False))  # 30 dias
        
        print(f"✅ Cache salvo: {question_hash}")
        return question_hash

    def get_qa(self, question: str) -> Optional[Dict]:
        """
        Busca pergunta no cache
        """
        question_hash = self.hash_question(question)
        key = f"sql_cache:{question_hash}"
        
        result = self.redis.get(key)
        if result:
            return json.loads(result)
        return None

    def list_all_cache(self) -> List[Dict]:
        """
        Lista todo o cache de SQL
        """
        keys = self.redis.keys("sql_cache:*")
        results = []
        
        for key in keys:
            data = self.redis.get(key)
            if data:
                results.append(json.loads(data))
        return results

    def delete_cache(self, question: str) -> bool:
        """
        Deleta entrada do cache
        """
        question_hash = self.hash_question(question)
        key = f"sql_cache:{question_hash}"
        return bool(self.redis.delete(key))

    def clear_all_cache(self) -> bool:
        """
        Deleta todo o cache do banco de dados Redis atual.
        Retorna True se a operação for bem-sucedida.
        """
        try:
            self.redis.flushdb()
            print("✅ Todo o cache do Redis foi apagado com sucesso.")
            return True
        except Exception as e:
            print(f"❌ Erro ao apagar o cache do Redis: {e}")
            return False


In [6]:
manager = RedisManager()
redis_client = manager.connect()

if redis_client:
    sql_cache = SQLCache(redis_client=redis_client)

In [7]:
cache_data = [
    {
        "id": 1,
        "pergunta": "Os 10 países que mais produziram filmes e o ano em que a maior parte foi produzido",
        "sql_code":  "WITH top_countries AS ( SELECT country, COUNT(*) AS total_movies FROM netflix.netflix WHERE type = 'Movie' AND country IS NOT NULL GROUP BY country ORDER BY total_movies DESC LIMIT 10 ), country_year_counts AS ( SELECT country, release_year, COUNT(*) AS year_count FROM netflix.netflix WHERE type = 'Movie' AND country IS NOT NULL GROUP BY country, release_year ) SELECT tc.country, tc.total_movies, cyc.release_year AS year_most_movies, cyc.year_count AS movies_in_that_year FROM top_countries tc JOIN country_year_counts cyc ON tc.country = cyc.country JOIN ( SELECT country, MAX(year_count) AS max_year_count FROM country_year_counts GROUP BY country ) max_cyc ON cyc.country = max_cyc.country AND cyc.year_count = max_cyc.max_year_count ORDER BY tc.total_movies DESC, tc.country;",
        "explicacao": "A consulta identifica os 10 países que mais produziram filmes (type = 'Movie') na tabela netflix.netflix, contando o número de filmes por país. Em seguida, para cada um desses países, determina o ano (release_year) em que o país mais produziu filmes, retornando o país, a quantidade total de filmes produzidos e o ano de maior produção junto com a quantidade de filmes desse ano. Utiliza subconsultas e funções de agregação para obter os resultados desejados.",
        "python_code": "df['column_0'] = df['column_0'].fillna('').astype(str) df['column_1'] = df['column_1'].fillna(0) df['column_2'] = df['column_2'].fillna(0) df['column_3'] = df['column_3'].fillna(0) df_viz = df.rename(columns={ 'column_0': 'country', 'column_1': 'total_movies', 'column_2': 'year_most_movies', 'column_3': 'movies_in_that_year' })",
        "tag": "paises, filmes",
        "data_criacao": "2025-01-01",
        "data_atualizacao": "2025-01-01",
        "status": "ativo",
    },
    {
        "id": 2,
        "pergunta": "Quero uma visualização dos 10 países que mais produziram filmes e o ano em que a maior parte foi produzido",
        "sql_code": "SELECT country, total_filmes, ano_maior_producao FROM ( SELECT country, COUNT(*) AS total_filmes, (SELECT release_year FROM ( SELECT release_year, COUNT(*) AS filmes_por_ano FROM netflix.netflix n2 WHERE n2.type = 'Movie' AND FIND_IN_SET(country, n2.country) GROUP BY release_year ORDER BY filmes_por_ano DESC, release_year DESC LIMIT 1 ) ano_max ) AS ano_maior_producao FROM ( SELECT TRIM(SUBSTRING_INDEX(SUBSTRING_INDEX(country, ',', numbers.n), ',', -1)) AS country, release_year FROM netflix.netflix JOIN ( SELECT 1 n UNION SELECT 2 UNION SELECT 3 UNION SELECT 4 UNION SELECT 5 UNION SELECT 6 UNION SELECT 7 UNION SELECT 8 UNION SELECT 9 UNION SELECT 10 ) numbers ON CHAR_LENGTH(country) - CHAR_LENGTH(REPLACE(country, ',', '')) >= numbers.n - 1 WHERE type = 'Movie' AND country IS NOT NULL AND country <> '' ) countries GROUP BY country ORDER BY total_filmes DESC LIMIT 10 ) top_countries;",
        "explicacao": "A consulta identifica os 10 países que mais produziram filmes (type = 'Movie') na tabela netflix.netflix, considerando que o campo 'country' pode conter múltiplos países separados por vírgula. Para cada país, conta o número de filmes produzidos e determina o ano em que mais filmes foram lançados (release_year) para esse país. Utiliza subconsultas e funções de string para separar países, agrupar por país e ano, e selecionar o ano de maior produção para cada país. O resultado traz os 10 países com maior produção de filmes e o respectivo ano de maior produção.",
        "python_code": "import pandas as pd import plotly.graph_objects as go # Tratamento dos dados # Renomeando as colunas para nomes mais descritivos # Supondo que column_0 = 'country', column_1 = 'total_filmes', column_2 = 'ano_maior_producao' df['country'] = df['column_0'].fillna('').astype(str) df['total_filmes'] = pd.to_numeric(df['column_1'], errors='coerce').fillna(0) df['ano_maior_producao'] = df['column_2'].fillna(0).astype(str) # Selecionando os 10 países com maior produção de filmes df_top10 = df.sort_values('total_filmes', ascending=False).head(10) # Criando o gráfico de barras fig = go.Figure() fig.add_trace(go.Bar( x=df_top10['country'], y=df_top10['total_filmes'], text=df_top10['ano_maior_producao'], textposition='outside', marker_color='indianred' )) fig.update_layout( title='Top 10 Países em Produção de Filmes e Ano de Maior Produção', xaxis_title='País', yaxis_title='Total de Filmes Produzidos', xaxis_tickangle=45, uniformtext_minsize=8, uniformtext_mode='hide', bargap=0.2, plot_bgcolor='white' ) ",
        "tag": "paises, filme",
        "data_criacao": "2025-01-01",
        "data_atualizacao": "2025-01-01",
        "status": "ativo"
    },
    {
        "id": 3,
        "pergunta": "Um gráfico de linhas mostrando o número de filmes lançados por ano a partir de 2000",
        "sql_code": "SELECT release_year, COUNT(*) AS release_year FROM netflix.netflix WHERE type = 'Movie' AND release_year >= 2000 GROUP BY release_year ORDER BY release_year ASC;",
        "explicacao": "A consulta retorna o número de filmes lançados por ano a partir de 2000. Utiliza a coluna 'type' para filtrar apenas os registros que são filmes ('Movie') e a coluna 'release_year' para agrupar e contar os filmes lançados em cada ano. O filtro WHERE garante que apenas anos a partir de 2000 sejam considerados. O resultado é agrupado por 'release_year' e ordenado de forma crescente para facilitar a visualização em um gráfico de linhas.",
        "python_code": "import pandas as pd import plotly.graph_objects as go # Supondo que o DataFrame já está carregado na variável df # Renomeando as colunas para refletir os nomes corretos # Se necessário, ajuste para os nomes reais do seu DataFrame df = df.rename(columns={'column_0': 'release_year', 'column_1': 'num_filmes'}) df['release_year'] = pd.to_datetime(df['release_year'], format='%Y').dt.year df = df.sort_values('release_year') fig = go.Figure() fig.add_trace(go.Scatter(x=df['release_year'], y=df['num_filmes'], mode='lines+markers', line=dict(color='royalblue'), marker=dict(size=8))) fig.update_layout( title='Número de filmes lançados por ano na Netflix (a partir de 2000)', xaxis_title='Ano de lançamento', yaxis_title='Número de filmes lançados', xaxis=dict(tickmode='linear'), template='plotly_white', margin=dict(l=40, r=40, t=60, b=40) ) # O sistema exibirá automaticamente o gráfico, não é necessário usar fig.show()",
        "tag": "filmes, ano",
        "data_criacao": "2025-01-01",
        "data_atualizacao": "2025-01-01",
        "status": "ativo"
    },
    {
        "id": 4,
        "pergunta": "Quantos filmes os Estados Unidos produziu em 2017?",
        "sql_code": "SELECT COUNT(*) AS release_year FROM netflix.netflix WHERE type = 'Movie' AND release_year = 2017 AND country LIKE '%United States%';",
        "explicacao": "A consulta conta o número de filmes ('type' = 'Movie') produzidos pelos Estados Unidos ('country' LIKE '%United States%') e lançados em 2017 ('release_year' = 2017) na tabela netflix.netflix. O filtro LIKE é utilizado para garantir que registros onde 'country' contenha 'United States' sejam considerados, mesmo que haja múltiplos países listados.",
        "python_code": "",
        "tag": "filmes",
        "data_criacao": "2025-01-01",
        "data_atualizacao": "2025-01-01",
        "status": "Ativo"
    },

]

In [8]:
import json

json_data = json.dumps(cache_data)
print(json_data)

[{"id": 1, "pergunta": "Os 10 pa\u00edses que mais produziram filmes e o ano em que a maior parte foi produzido", "sql_code": "WITH top_countries AS ( SELECT country, COUNT(*) AS total_movies FROM netflix.netflix WHERE type = 'Movie' AND country IS NOT NULL GROUP BY country ORDER BY total_movies DESC LIMIT 10 ), country_year_counts AS ( SELECT country, release_year, COUNT(*) AS year_count FROM netflix.netflix WHERE type = 'Movie' AND country IS NOT NULL GROUP BY country, release_year ) SELECT tc.country, tc.total_movies, cyc.release_year AS year_most_movies, cyc.year_count AS movies_in_that_year FROM top_countries tc JOIN country_year_counts cyc ON tc.country = cyc.country JOIN ( SELECT country, MAX(year_count) AS max_year_count FROM country_year_counts GROUP BY country ) max_cyc ON cyc.country = max_cyc.country AND cyc.year_count = max_cyc.max_year_count ORDER BY tc.total_movies DESC, tc.country;", "explicacao": "A consulta identifica os 10 pa\u00edses que mais produziram filmes (type

In [9]:
for data in cache_data:
    print(data)

{'id': 1, 'pergunta': 'Os 10 países que mais produziram filmes e o ano em que a maior parte foi produzido', 'sql_code': "WITH top_countries AS ( SELECT country, COUNT(*) AS total_movies FROM netflix.netflix WHERE type = 'Movie' AND country IS NOT NULL GROUP BY country ORDER BY total_movies DESC LIMIT 10 ), country_year_counts AS ( SELECT country, release_year, COUNT(*) AS year_count FROM netflix.netflix WHERE type = 'Movie' AND country IS NOT NULL GROUP BY country, release_year ) SELECT tc.country, tc.total_movies, cyc.release_year AS year_most_movies, cyc.year_count AS movies_in_that_year FROM top_countries tc JOIN country_year_counts cyc ON tc.country = cyc.country JOIN ( SELECT country, MAX(year_count) AS max_year_count FROM country_year_counts GROUP BY country ) max_cyc ON cyc.country = max_cyc.country AND cyc.year_count = max_cyc.max_year_count ORDER BY tc.total_movies DESC, tc.country;", 'explicacao': "A consulta identifica os 10 países que mais produziram filmes (type = 'Movie')

## Armazenar cachê semantico

In [10]:
from redisvl.extensions.cache.llm import SemanticCache
from langchain_openai import AzureOpenAIEmbeddings
from redisvl.utils.vectorize import AzureOpenAITextVectorizer
import os
from dotenv import load_dotenv

load_dotenv()

api_key=os.getenv("AZURE_OPENAI_EMBEDDINGS")
endpoint=os.getenv("AZURE_EMBEDDINGS_ENDPOINT")
api_version=os.getenv("OPENAI_API_VERSION")

vectorizer = AzureOpenAITextVectorizer(
    # deployment_name="text-embedding-ada-002",
    api_config={
    "api_key":api_key,
    "azure_endpoint":endpoint,
    "api_version":api_version
    }
)

# test = embeddings.embed("This is a test sentence.")
# print("Vector dimensions: ", len(test))
# test[:10]

c:\AI\bi_agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from redisvl.extensions.cache.llm import SemanticCache

# === 1. CRIAR/CONECTAR AO CACHE ===

# OPÇÃO 1: SEM EXPIRAÇÃO (Recomendado)
cache = SemanticCache(
    name="cache-index",
    vectorizer=vectorizer,
    ttl=None,  # ← None = sem expiração
    redis_url=f"redis://{REDIS_HOST}:{REDIS_PORT}",
    distance_threshold=0.2,
    overwrite=True  
)

for data in cache_data:
    question = data.get('pergunta')
    explanation = data.get('explicacao')
    query = data.get('sql_code')
    
    # Formatar a resposta
    response = f"""Explicação:
    {explanation}

    Query SQL:
    {query}"""

    metadata = {
        "id": str(data.get('id', '')),
        # "departamento": data.get('departamento', ''),
        # "fonte": data.get('fonte', '')
    }
    

    cache.store(
            prompt=question,
            response=response,
            metadata=metadata
        )

print(f"✅ {len(cache_data)} perguntas armazenadas no cache!")

✅ 4 perguntas armazenadas no cache!


# Cache exato

In [13]:

for data in cache_data:
    # 1. Pega os campos obrigatórios
    question = data.get("question", "")
    querye = data.get("querye", "")    
    
    metadata = {k: v for k, v in data.items() if k not in ["question", "querye"]}
    # print(metadata)
    # 4. Chama a função
    try:
        sql_cache.store_qa(question=question, querye=querye, metadata=metadata)
        print(f"✅ Salvo: {question[:50]}...")
    except Exception as e:
        print(f"❌ Erro ao salvar: {e}")

✅ Cache salvo: d41d8cd98f00
✅ Salvo: ...
✅ Cache salvo: d41d8cd98f00
✅ Salvo: ...
✅ Cache salvo: d41d8cd98f00
✅ Salvo: ...
✅ Cache salvo: d41d8cd98f00
✅ Salvo: ...


## Teste Cache semantico

In [ ]:
user_question = "Quantos filmes a India produziu?"
result = cache.check(prompt=user_question, num_results=3,distance_threshold=0.2)

if result:
    print(f"\n🎯 Cache encontrado!")
    print(f"\nPergunta original: {result[0]['prompt']}")
    print(f"\n{result[0]['response']}")
    print(f"\nMetadados: {result[0].get('metadata', {})}")
else:
    print(f"\n❌ Nenhum cache encontrado para: '{user_question}'")


🎯 Cache encontrado!

Pergunta original: Um gráfico de linhas mostrando o número de filmes lançados por ano a partir de 2000

Explicação:
    A consulta retorna o número de filmes lançados por ano a partir de 2000. Utiliza a coluna 'type' para filtrar apenas os registros que são filmes ('Movie') e a coluna 'release_year' para agrupar e contar os filmes lançados em cada ano. O filtro WHERE garante que apenas anos a partir de 2000 sejam considerados. O resultado é agrupado por 'release_year' e ordenado de forma crescente para facilitar a visualização em um gráfico de linhas.

    Query SQL:
    SELECT release_year, COUNT(*) AS release_year FROM netflix.netflix WHERE type = 'Movie' AND release_year >= 2000 GROUP BY release_year ORDER BY release_year ASC;

Metadados: {'id': '3'}


In [ ]:
cache.check(prompt= "Quantos filmes a India produziu?")

[]

# Teste cache exato

In [ ]:
todos_os_dados = sql_cache.list_all_cache()
todos_os_dados